# Example 11.2 / 2: Starting Values

Companion notebook for **11.2 Nonlinear Models** in the AMPL lab Documentation, section "Starting Values". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Starting Values

The `:=` phrase sets `.init0`, the original initial guess. The current guess is available as
`.init`, while `.val` is the current variable value and holds the returned solution after a solve.
The solver may use the current guess as its starting point. Different starts can lead to different
stationary points in a nonconvex model, and a poor start can cause an evaluation or convergence
failure.

When no value is supplied, AMPL tentatively initializes a variable to zero. A solver may replace
uninitialized values, while retaining explicitly initialized ones. Zero is unsafe for expressions
such as `log(x)`, `1 / x`, and derivatives of `sqrt(x)`, so initialize each variable at a point
where every expression the solver must evaluate is defined. Initial dual values can also be
assigned through constraint values, although only solvers that support multiplier starts use them.

The suffixes on one variable, before and after a solve. `Spend` is declared with `:= 2`, then
moved to `1` before solving; `Idle` has no initial value; the constraint `Cap` declares an
initial dual value with `:= 0.5`. The amplpy methods `init0()`, `init()`, `value()` and
`dinit0()` read the same suffixes:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var Spend >= 0, := 2;  # declared start 2 is kept in Spend.init0
var Idle;              # no initial value declared

maximize Net: 5 * log(1 + Spend) - Spend + 0 * Idle;  # concave net benefit; Idle has zero weight

subject to Cap := 0.5: Spend <= 10;  # initial dual value 0.5 (Cap.dinit0)

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
def show(label):
    s = ampl.var["Spend"]
    print(label, s.init0(), s.init(), round(s.value(), 6))

show("declared:")
ampl.var["Spend"].set_value(1)
show("after set_value(1):")
ampl.solve(solver="ipopt", verbose=False)
show("after solve:")
print("Idle:", ampl.var["Idle"].value(), " Cap.dinit0:", ampl.con["Cap"].dinit0())

declared: 2.0 2.0 2.0
after set_value(1): 2.0 2.0 1.0
after solve: 2.0 1.0 4.0
Idle: 0.0  Cap.dinit0: 0.5


### Pure AMPL: model.run

Pure AMPL, with `let` in place of `set_value()`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver ipopt;
display Spend.init0, Spend.init, Spend.val;
let Spend := 1;
display Spend.init0, Spend.init, Spend.val;
solve > ipopt.log;
display Spend.init0, Spend.init, Spend.val, Idle, Cap.dinit0;

Spend.init0 = 2
Spend.init = 2
Spend.val = 2

Spend.init0 = 2
Spend.init = 2
Spend.val = 1

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
Spend.init0 = 2
Spend.init = 1
Spend.val = 4
Idle = 0
Cap.dinit0 = 0.5

